# Your mosaic in a Game of Life simulator

A mosaic is not just a picture of a still life: it *is* one. This notebook exports the pattern to a `.cells` file for [Golly](https://golly.sourceforge.io/), the standard Game of Life simulator, where it stays frozen generation after generation, until a glider flies in and the whole portrait starts to unravel.

Only the still life is exported. The coloured background (the elementary cellular automaton) is decoration and would not be stable.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from gol_mosaics import GollyExporter, MosaicGenerator
from gol_mosaics.life import is_still_life, life_step

from pathlib import Path

# The repository root, wherever Jupyter was started from
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").exists())


## Make a mosaic, and keep its cells

`return_arrays=True` returns the still life itself (a 0/1 array, one entry per cell) next to the rendered image.

In [ ]:
image = Image.open(REPO / 'input/images/john.png')

generator = MosaicGenerator(level=4, grid_size=40, eca_rule=30)
picture, cells, background = generator.generate_from_pil(
    image, seed=1, return_arrays=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(picture)
axes[0].set_title('rendered mosaic')
axes[1].imshow(cells, cmap='binary', interpolation='nearest')
axes[1].set_title(f'the still life: {cells.shape[1]} x {cells.shape[0]} cells, '
                  f'{int(cells.sum()):,} alive')
for ax in axes:
    ax.axis('off')
plt.show()

## Check that it is a still life

One step of the Game of Life must leave every cell as it is.

In [ ]:
print('still life:', is_still_life(cells))

## Export, with a glider

`add_glider` places a glider in one corner, heading into the grid. In Golly, press play: the portrait holds still until the glider hits it.

In [ ]:
path = REPO / 'output/golly/my_mosaic.cells'
GollyExporter.export_to_cells(cells, str(path), add_glider='bottom right')
print(f'saved {path.relative_to(REPO)} ({path.stat().st_size / 1024:.0f} kB)')

## What the glider does

The same collision, simulated here on the bottom-right corner of the pattern: the glider travels up and to the left and the still life starts to break where it lands.

In [ ]:
corner = np.pad(cells[-60:, -60:], ((0, 12), (0, 12)))
corner = GollyExporter._add_glider_pattern(corner, 'bottom right')
frames = [corner]
for _ in range(160):
    frames.append(life_step(frames[-1]))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, t in zip(axes, (0, 40, 100, 160)):
    ax.imshow(frames[t], cmap='binary', interpolation='nearest')
    ax.set_title(f'generation {t}')
    ax.axis('off')
plt.show()